In [ ]:
from PySAM import ResourceTools as tools

building_id = ["46496","112157","111549","126745","305","442412","404322","448494","67263","168160","1652","467842"]
latitude = ["33.56","35.14","32.87","39.81","41.88","44.85","40.68","39.99","45.59","35.22","44.47","42.95"]
longitude = ["-86.75","-111.67","-117.15","-105.14","-71.02","-93.03","-74.17","-82.88","-122.6","-101.71","-73.15","-87.9"]
years = [2018, 2019, 2020, 2021, 2022, 2023]

sam_api_key = "<API_KEY>"
sam_email = "<EMAIL>"



for i, lat in enumerate(latitude):
    for year in years:

        building_path = "building_models\\bldg" + building_id[i] + "-up00\\weather_data"

        nsrdbfetcher = tools.FetchResourceFiles(
                        tech='solar',
                        nrel_api_key=sam_api_key,
                        nrel_api_email=sam_email,
                        resource_type='nsrdb-GOES-aggregated-v4-0-0',
                        resource_year=str(year),
                        resource_dir=building_path)

        lon_lats = []

        #for index, row in locs.iterrows():
        lon_lat = (float(longitude[i]), float(lat))
        lon_lats.append(lon_lat)

        nsrdbfetcher.fetch(lon_lats)


Starting data download for solar using 1 thread workers.
Getting list of available NSRDB files for 33.56, -86.75.
List of available data saved to building_models\bldg46496-up00\weather_data/nsrdb_data_query_response_33.56_-86.75.json.
https://developer.nrel.gov/api/nsrdb/v2/solar/nsrdb-GOES-aggregated-v4-0-0-download.csv?names=2018&wkt=POINT%28-86.75+33.56%29&interval=60&api_key=RkdVzEGa7mlEGqV9jzwjhrh95ACZOr6PFr0K0yTc&email=brian.mirletz@nrel.gov&utc=false
Success! File downloaded to building_models\bldg46496-up00\weather_data\nsrdb_33.56_-86.75_nsrdb-GOES-aggregated-v4-0-0_60_2018.csv.

Starting data download for solar using 1 thread workers.
File already exists. Skipping download: building_models\bldg46496-up00\weather_data\nsrdb_33.56_-86.75_nsrdb-GOES-aggregated-v4-0-0_60_2019.csv

Starting data download for solar using 1 thread workers.
Getting list of available NSRDB files for 33.56, -86.75.
List of available data saved to building_models\bldg46496-up00\weather_data/nsrdb_data_

In [ ]:
import pandas as pd
import csv

for k, file_lat in enumerate(latitude):
    for year in years:
        
        """
        NSRDB CSV to EPW conversion code courtesy of Eric Bonnema, https://github.com/bonnema
        """

        folder_name = "building_models\\bldg" + str(building_id[k]) + "-up00\\weather_data\\"
        # read sam csv in two dataframes
        sam_name = folder_name + 'nsrdb_' + str(file_lat) + '_' + str(longitude[k]) + '_nsrdb-GOES-aggregated-v4-0-0_60_' + str(year)
        filename = sam_name + ".csv"
        info_df = pd.read_csv(filename, nrows=1)
        data_df = pd.read_csv(filename, skiprows=2)

        # get info for epw header
        city = info_df['City'].values[0]
        state = info_df['State'].values[0]
        cntry = info_df['Country'].values[0]
        src = info_df['Source'].values[0]
        locn_id = info_df['Location ID'].values[0]
        lat = info_df['Latitude'].values[0]
        lon = info_df['Longitude'].values[0]
        tz = info_df['Time Zone'].values[0]
        ele = info_df['Elevation'].values[0]

        # write epw header
        epw_file = open(sam_name + '.epw', 'w', newline='\n', encoding='utf-8')
        epw_file.write(f'LOCATION,{city},{state},{cntry},{src},{locn_id},{lat},{lon},{tz},{ele}\n')
        epw_file.write('DESIGN CONDITIONS,0\n')
        epw_file.write('TYPICAL/EXTREME PERIODS,0\n')
        epw_file.write('GROUND TEMPERATURES,0\n')
        epw_file.write('HOLIDAYS/DAYLIGHT SAVINGS,No,0,0,0\n')
        epw_file.write('COMMENTS 1,\n')
        epw_file.write('COMMENTS 2,\n')
        epw_file.write(f'DATA PERIODS,1,1,Data,Sunday,1/1,12/31\n')

        # define epw writer
        writer = csv.writer(epw_file, delimiter=',', lineterminator='\n')

        # write each row of data df in epw format
        for i in range(data_df.shape[0]):

            # calculate opaque sky cover
            if data_df['Clearsky DNI'][i] == 0:
                osc = 5
            else:
                osc = data_df['DNI'][i] / data_df['Clearsky DNI'][i]

            # build epw row
            row = [
                data_df['Year'][i],
                data_df['Month'][i],
                data_df['Day'][i],
                data_df['Hour'][i] + 1, # e+ expects hour to start at 1
                data_df['Minute'][i] - 30, # e+ expects minute to start at 0
                '?9?9?9?9E0?9?9?9?9?9?9?9?9?9?9?9?9?9?9?9*9*9?9*9*9',
                data_df['Temperature'][i],
                data_df['Dew Point'][i],
                data_df['Relative Humidity'][i],
                data_df['Pressure'][i] * 100, # convert mbar to pa
                '9999',
                '9999',
                '9999',
                data_df['GHI'][i],
                data_df['DNI'][i],
                data_df['DHI'][i],
                '999999',
                osc,
                '999999',
                '9999',
                data_df['Wind Direction'][i],
                data_df['Wind Speed'][i],
                '99',
                '99',
                '9999',
                '99999',
                '9',
                '999999999',
                '999',
                '0.999',
                '0',
                '88',
                data_df['Surface Albedo'][i],
                '999',
                '99'
            ]

            # write epw row
            writer.writerow(row)

        # close epw
        epw_file.close()
